# Hinglish to English Machine Translation Pipeline

**Prompting mode:** Zero-shot (no in-context examples)  
**Input column:** `cm_text_r` (Romanized Hinglish)  
**Output column:** `predicted_english` (translated English)  
**Reference column:** `emb_text` (ground truth English)

## Workflow — change provider and re-run ALL cells
Edit **only Cell 3 (Config)** to swap provider/model/key, then use
`Runtime → Run all` and every cell will pick up the new settings.

### Supported API Providers
| Provider | Models | Notes |
|---|---|---|
| **Groq** | llama-3.1-8b-instant, llama-3.3-70b-versatile, mixtral-8x7b-32768 | Fastest, free tier |
| **Together AI** | Meta-Llama-3.1-8B, Mistral-7B, Qwen2-72B | Good free credits |
| **OpenRouter** | Any open-source model | Unified API, pay-per-use |

### Data Format Expected
```json
{"emb_text": "English ground truth", "cm_text_r": "isamen kuchha provisions hain jo ..."}
```
Supports `.csv`, `.json`, and `.jsonl` input files.

In [ ]:
# Cell 1: Install dependencies
!pip install groq pandas tqdm sacrebleu -q

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — ONLY CELL YOU NEED TO EDIT.  Change values here,      ║
# ║  then Runtime → Run all to apply to every downstream cell.       ║
# ╚══════════════════════════════════════════════════════════════════╝
import os

# ── Provider & Credentials ───────────────────────────────────────────────
# Choose one: "groq" | "together" | "openrouter"
API_PROVIDER = "groq"

# Paste your key below (get from the links in the comments)
#   Groq:       https://console.groq.com/keys
#   Together:   https://api.together.xyz/settings/api-keys
#   OpenRouter: https://openrouter.ai/keys
API_KEY = "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"

# ── Model ────────────────────────────────────────────────────────────────
# Groq:       "llama-3.1-8b-instant" | "llama-3.3-70b-versatile" | "mixtral-8x7b-32768"
# Together:   "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo" | "mistralai/Mistral-7B-Instruct-v0.3"
# OpenRouter: "meta-llama/llama-3.1-8b-instruct:free" | "mistralai/mistral-7b-instruct:free"
MODEL = "llama-3.1-8b-instant"

# ── Prompting Mode ───────────────────────────────────────────────────────
# "zero_shot" : only the system instruction, no examples  (current setting)
# "few_shot"  : prepend 5 curated examples before each input
PROMPT_MODE = "zero_shot"

# ── File Paths ───────────────────────────────────────────────────────────
DRIVE_DATA_PATH    = '/content/drive/MyDrive/mBART_Finetune_Hinglish/pacman/cleaned_data'
data_DIR           = '/content/drive/MyDrive/mBART_Finetune_Hinglish/pacman/bidirect'
TOKENIZED_DATA_DIR = os.path.join(data_DIR, 'tokenized_data')

INPUT_FILE  = os.path.join(DRIVE_DATA_PATH, "train/Final_6M_CM_SelectedColumns_train.jsonl")
# Output path is auto-derived from provider+model so each run saves separately:
_model_slug = MODEL.replace('/', '_').replace('.', '_').replace('-', '_')
OUTPUT_FILE = f'/content/drive/MyDrive/llm/{API_PROVIDER}/{_model_slug}/output_translated.csv'

# ── Column Names ─────────────────────────────────────────────────────────
INPUT_COL     = "cm_text_r"          # Romanized Hinglish input
REFERENCE_COL = "emb_text"           # Ground truth English (for BLEU eval)
OUTPUT_COL    = "predicted_english"  # New column for translations

# ── Processing Settings ──────────────────────────────────────────────────
SAVE_EVERY          = 50    # Save progress every N rows
SLEEP_TIME          = 0.1   # Seconds between requests (increase if rate-limited)
MAX_RETRIES         = 3     # Retries on API errors
TEMPERATURE         = 0     # 0 = deterministic (best for translation)
MAX_TOKENS          = 300   # Max output tokens per translation
MAX_ROWS_TO_PROCESS = 1000  # Set to None to process all rows

print(f"Provider    : {API_PROVIDER}")
print(f"Model       : {MODEL}")
print(f"Prompt mode : {PROMPT_MODE}")
print(f"Input       : {INPUT_FILE}")
print(f"Output      : {OUTPUT_FILE}")


In [ ]:
# List files in the data directory
import os

print(f"Files in {DRIVE_DATA_PATH}:")
for root, dirs, files in os.walk(DRIVE_DATA_PATH):
    for name in files:
        print(os.path.join(root, name))


In [ ]:
# Load a sample of the input file and display column names
import pandas as pd
import json
import os

def load_sample_data(path, sample_rows=5):
    ext = os.path.splitext(path)[-1].lower()
    if ext == ".jsonl":
        with open(path, "r", encoding="utf-8") as f:
            # Read only a few lines for column inspection
            lines = [next(f) for _ in range(sample_rows)]
        records = [json.loads(line) for line in lines]
        return pd.DataFrame(records)
    elif ext == ".csv":
        return pd.read_csv(path, nrows=sample_rows)
    elif ext == ".json":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return pd.DataFrame(data).head(sample_rows)
        else:
            return pd.DataFrame([data])
    else:
        raise ValueError(f"Unsupported format: {ext}")

try:
    sample_df = load_sample_data(INPUT_FILE)
    print(f"Successfully loaded a sample of {INPUT_FILE}.")
    print("Column names:")
    for col in sample_df.columns:
        print(f"- {col}")
    print("\nFirst 5 rows of the sample data:")
    print(sample_df.head())
except Exception as e:
    print(f"Error loading sample data from {INPUT_FILE}: {e}")

In [ ]:
# Cell 4: Initialize API Client
import os, time, json
import pandas as pd
from tqdm.notebook import tqdm

def get_client():
    if API_PROVIDER == "groq":
        from groq import Groq
        return Groq(api_key=API_KEY)
    elif API_PROVIDER == "together":
        import subprocess
        subprocess.run(["pip", "install", "together", "-q"])
        from together import Together
        return Together(api_key=API_KEY)
    elif API_PROVIDER == "openrouter":
        import subprocess
        subprocess.run(["pip", "install", "openai", "-q"])
        from openai import OpenAI
        return OpenAI(api_key=API_KEY, base_url="https://openrouter.ai/api/v1")
    else:
        raise ValueError(f"Unknown provider: {API_PROVIDER}")

client = get_client()
print(f"{API_PROVIDER.upper()} client initialized successfully")

In [ ]:
# Cell 5: Translation Prompt and Function
#
# PROMPT_MODE (set in Cell 3) controls whether examples are sent:
#   "zero_shot" — system instruction only, no examples
#   "few_shot"  — 5 curated examples prepended to every request

SYSTEM_PROMPT = """You are a strict translation system specializing in Hinglish to English translation.
Hinglish is Romanized Hindi-English code-mixed text (Hindi words written in Latin script mixed with English words).

Rules:
- Translate into natural, fluent, grammatically correct English
- Preserve the EXACT original meaning
- Hindi words must be translated; English words kept as-is
- Use proper casing and punctuation
- Output ONLY the final English sentence — no explanation, no extra text"""

# Few-shot examples (only used when PROMPT_MODE == "few_shot")
FEW_SHOT_EXAMPLES = """Examples:
Input: yah product bahut accha hai aur price bhi reasonable hai
Output: This product is very good and the price is also reasonable.

Input: mujhe yah charger pasand nahi hai kyunki yah bahut slow charge karta hai
Output: I don't like this charger because it charges very slowly.

Input: delivery fast thi aur packaging bhi bahut achi thi, highly recommend karunga
Output: The delivery was fast and the packaging was also very good, I would highly recommend it.

Input: isamen kuchha provisions hain jo LGBT Pakistani citizens ke constitutional rights ko prabhaavit kar sakate hain
Output: It does contain certain provisions that may impact the constitutional rights of LGBT Pakistani citizens.

Input: battery life bahut achi hai, ek charge mein poora din chalta hai
Output: The battery life is very good, it lasts the whole day on a single charge."""


def build_user_prompt(text):
    """Build the user-turn prompt according to PROMPT_MODE."""
    if PROMPT_MODE == "few_shot":
        return FEW_SHOT_EXAMPLES + f"\n\nInput: {text}\nOutput:"
    else:  # zero_shot (default)
        return f"Translate the following Hinglish text to English.\n\nInput: {text}\nOutput:"


def translate(text, retries=MAX_RETRIES):
    prompt = build_user_prompt(text)
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt}
                ],
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS
            )
            result = response.choices[0].message.content.strip().strip('"').strip("'")
            if result.lower().startswith("output:"):
                result = result[7:].strip()
            return result
        except Exception as e:
            err = str(e)
            if "rate_limit" in err.lower() or "429" in err:
                wait = 60 * (attempt + 1)
                print(f"Rate limit. Waiting {wait}s (retry {attempt+1}/{retries})...")
                import time; time.sleep(wait)
            elif attempt < retries - 1:
                print(f"Error attempt {attempt+1}: {e}")
                import time; time.sleep(5)
            else:
                print(f"Failed after {retries} retries: {e}")
                return "TRANSLATION_ERROR"
    return "TRANSLATION_ERROR"


# Quick sanity test
test = "yah product bahut accha hai aur bahut fast delivery aayi thi"
print(f"Prompt mode : {PROMPT_MODE}")
print(f"Test input  : {test}")
print(f"Translation : {translate(test)}")


In [ ]:
# Cell 6: Load Data (supports .json, .jsonl, .csv)

def load_data(path):
    ext = os.path.splitext(path)[-1].lower()
    if ext == ".json":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        df = pd.DataFrame(data if isinstance(data, list) else [data])
    elif ext == ".jsonl":
        records = []
        with open(path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if MAX_ROWS_TO_PROCESS is not None and i >= MAX_ROWS_TO_PROCESS:
                    break
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        df = pd.DataFrame(records)
    elif ext == ".csv":
        df = pd.read_csv(path, nrows=MAX_ROWS_TO_PROCESS)
    else:
        raise ValueError(f"Unsupported format: {ext}")
    return df

def prepare_df():
    # Data is loaded from the path specified in INPUT_FILE, which is configured in Cell 3.
    input_df = load_data(INPUT_FILE)
    print(f"Loaded {len(input_df):,} rows | Columns: {list(input_df.columns)}")

    if os.path.exists(OUTPUT_FILE):
        existing = pd.read_csv(OUTPUT_FILE)
        done = existing[OUTPUT_COL].notna().sum() if OUTPUT_COL in existing.columns else 0
        print(f"Resuming: {done:,} rows already translated")
        input_df[OUTPUT_COL] = ""
        if OUTPUT_COL in existing.columns and len(existing) <= len(input_df):
            input_df.loc[:len(existing)-1, OUTPUT_COL] = existing[OUTPUT_COL].values
    else:
        input_df[OUTPUT_COL] = ""
        print("Starting fresh")

    input_df[OUTPUT_COL] = input_df[OUTPUT_COL].astype(object)
    return input_df

df = prepare_df()
done = df[OUTPUT_COL].apply(lambda x: str(x).strip() not in ["", "nan", "TRANSLATION_ERROR"]).sum()
print(f"Status: {done:,} done | {len(df)-done:,} remaining | {len(df):,} total")
df[[INPUT_COL, REFERENCE_COL, OUTPUT_COL]].head(3)

In [ ]:
# Cell 7: Run Translation Pipeline

def run_translation(df):
    # Ensure the output directory exists
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

    pending = [
        i for i, row in df.iterrows()
        if str(row[OUTPUT_COL]).strip() in ["", "nan", "TRANSLATION_ERROR"]
    ]
    print(f"Translating {len(pending):,} rows | Model: {MODEL} | Provider: {API_PROVIDER}")
    print(f"Saving every {SAVE_EVERY} rows to: {OUTPUT_FILE}")

    errors = 0
    t0 = time.time()

    for count, i in enumerate(tqdm(pending, desc="Translating"), 1):
        text = df.at[i, INPUT_COL]
        if pd.isna(text) or str(text).strip() == "":
            df.at[i, OUTPUT_COL] = ""
            continue

        result = translate(str(text))
        df.at[i, OUTPUT_COL] = result
        if result == "TRANSLATION_ERROR":
            errors += 1

        if count % SAVE_EVERY == 0:
            df.to_csv(OUTPUT_FILE, index=False)
            elapsed = time.time() - t0
            rate = count / elapsed
            eta = (len(pending) - count) / rate if rate > 0 else 0
            print(f"Checkpoint {count:,}/{len(pending):,} | {rate:.1f} rows/s | ETA: {eta/60:.1f} min")

        time.sleep(SLEEP_TIME)

    df.to_csv(OUTPUT_FILE, index=False)
    elapsed = time.time() - t0
    print(f"Done! Translated: {len(pending)-errors:,} | Errors: {errors} | Time: {elapsed/60:.1f} min")
    print(f"Saved to: {OUTPUT_FILE}")
    return df


df = run_translation(df)

In [ ]:
# Cell 8: Preview Results
pd.set_option("display.max_colwidth", 120)
sample = df[[INPUT_COL, REFERENCE_COL, OUTPUT_COL]].dropna(subset=[OUTPUT_COL]).head(10)
sample

In [ ]:
# Cell 9: BLEU / chrF / TER Evaluation (vs ground truth emb_text)
import sacrebleu

eval_df = df[
    df[OUTPUT_COL].notna() &
    df[REFERENCE_COL].notna() &
    (df[OUTPUT_COL] != "TRANSLATION_ERROR")
].copy()

if len(eval_df) == 0:
    print("No rows available for evaluation")
else:
    hyp = eval_df[OUTPUT_COL].astype(str).tolist()
    ref = [eval_df[REFERENCE_COL].astype(str).tolist()]
    print(f"Evaluation on {len(eval_df):,} rows | Model: {MODEL}")
    print(f"BLEU : {sacrebleu.corpus_bleu(hyp, ref).score:.2f}")
    print(f"chrF : {sacrebleu.corpus_chrf(hyp, ref).score:.2f}")
    print(f"TER  : {sacrebleu.corpus_ter(hyp, ref).score:.2f}  (lower is better)")

In [ ]:
# Cell 9b: Install BERTScore
!pip install bert_score -q

In [ ]:
# Cell 9c: BERTScore Evaluation
try:
    import bert_score
except ModuleNotFoundError:
    print("bert_score not found, installing...")
    !pip install bert_score -q
    import bert_score # Try importing again

if len(eval_df) == 0:
    print("No rows available for BERTScore evaluation")
else:
    print(f"BERTScore Evaluation on {len(eval_df):,} rows | Model: {MODEL}")
    # BertScore expects lists of strings
    # Use 'predicted_english' as it's the actual column from the last translation run.
    # The global OUTPUT_COL was changed by cell-switch but the data column remains 'predicted_english'
    hypotheses = eval_df['predicted_english'].astype(str).tolist()
    references = eval_df[REFERENCE_COL].astype(str).tolist()

    # Calculate BERTScore (it can take some time to download the model the first time)
    P, R, F1 = bert_score.score(hypotheses, references, lang="en", verbose=True)

    print(f"BERTScore F1: {F1.mean().item():.2f}")
    print(f"BERTScore Precision: {P.mean().item():.2f}")
    print(f"BERTScore Recall: {R.mean().item():.2f}")

In [ ]:
# Cell 10: Retry Failed Translations
error_idx = df[df[OUTPUT_COL] == "TRANSLATION_ERROR"].index.tolist()
print(f"Found {len(error_idx)} error rows. Retrying...")

for i in tqdm(error_idx, desc="Retrying"):
    result = translate(str(df.at[i, INPUT_COL]))
    df.at[i, OUTPUT_COL] = result
    time.sleep(SLEEP_TIME)

df.to_csv(OUTPUT_FILE, index=False)
remaining = (df[OUTPUT_COL] == "TRANSLATION_ERROR").sum()
print(f"Done. Remaining errors: {remaining}")

## Switching Provider / Model

**You only ever need to edit Cell 3 (Config).** Change `API_PROVIDER`, `API_KEY`, and `MODEL` there, then do `Runtime → Run all`. Every cell below automatically reads those globals at call time, so:

- A new `client` is created with the new credentials (Cell 4)
- `OUTPUT_FILE` is re-derived from the new provider + model slug (Cell 3)
- Data is re-loaded and translation runs fresh (Cells 6 & 7)

**Common switch examples — paste into Cell 3:**

```python
# Together AI
API_PROVIDER = "together"
API_KEY      = "YOUR_TOGETHER_KEY"
MODEL        = "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo"

# OpenRouter
API_PROVIDER = "openrouter"
API_KEY      = "YOUR_OPENROUTER_KEY"
MODEL        = "meta-llama/llama-3.1-8b-instruct:free"

# Groq 70B
API_PROVIDER = "groq"
API_KEY      = "YOUR_GROQ_KEY"
MODEL        = "llama-3.3-70b-versatile"
```

Each run saves to a unique path like  
`/content/drive/MyDrive/llm/<provider>/<model_slug>/output_translated.csv`  
so results from different models never overwrite each other.


## Tips & Troubleshooting

**Rate Limits**
- Groq free tier: ~30 req/min for 8B models, ~14/min for 70B
- If rate-limited, set `SLEEP_TIME = 2.0` in Cell 3
- 1 lakh rows at 30 req/min ≈ 55 hours with 8B model

**Resuming After Crash**
- Re-run Cell 6 then Cell 7 — it auto-skips completed rows

**Comparing Models**
- Use a different `OUTPUT_COL` name per model run
- Use Cell 9 (BLEU) to compare quality

**Speed Tips**
- Use `llama-3.1-8b-instant` on Groq for maximum speed
- Together AI paid tier gives higher rate limits
- Run multiple Colab sessions on different row slices